# HAM10000 Edge-AI Compression Pipeline — Colab Runner

Run cells top to bottom. Requires: Colab GPU runtime (Runtime > Change runtime type > GPU), a Kaggle account, and `edge-ai-skin-lesion-project.zip` uploaded to this Colab session's file browser first.

Steps: install deps -> get Kaggle data -> unzip project code -> train baseline -> compress -> benchmark -> Pareto table + plot -> zip results for download.

In [ ]:
# 1. Install project dependencies (Colab ships TF already; this adds the rest)
!pip install -q tensorflow-model-optimization kaggle

In [ ]:
# 2. Upload your Kaggle API token (kaggle.json) when prompted
from google.colab import files
print("Upload kaggle.json (from kaggle.com -> Account -> Create New API Token)")
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# 3. Download HAM10000
!mkdir -p data/HAM10000
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p data/HAM10000 --unzip
!ls data/HAM10000

In [ ]:
# 4. Upload and unpack the project code zip (edge-ai-skin-lesion-project.zip)
print("Upload edge-ai-skin-lesion-project.zip")
files.upload()
!unzip -oq edge-ai-skin-lesion-project.zip
%cd edge-ai-skin-lesion
# point the project at the data you just downloaded (one level up)
!ln -sfn ../data/HAM10000 data/HAM10000
!ls data/HAM10000 | head

In [ ]:
# 5. Confirm config is set for a REAL run (imagenet weights, full epoch counts).
#    If a previous smoke test edited these, this cell resets them.
import importlib
from src import config
config.MOBILENET_WEIGHTS = "imagenet"
config.BASELINE_EPOCHS_FROZEN = 10
config.BASELINE_EPOCHS_FINETUNE = 15
config.PRUNE_EPOCHS = 8
config.QAT_EPOCHS = 5
config.PRUNING_TARGET_SPARSITY = [0.3, 0.5, 0.7]
config.BATCH_SIZE = 32
config.BENCHMARK_NUM_RUNS = 100
print("MOBILENET_WEIGHTS =", config.MOBILENET_WEIGHTS)

In [ ]:
# 6. Train baseline (frozen head -> fine-tune). ~15-30 min on Colab GPU depending on tier.
from src import train_baseline
train_baseline.main()

In [ ]:
# 7. Compression sweep: PTQ (dynamic + int8), pruning at 30/50/70%, pruning+int8, QAT+int8
from src import compress
compress.main()

In [ ]:
# 8. Benchmark every .tflite variant: size, CPU latency, accuracy, per-class F1
from src import benchmark
benchmark.main()

In [ ]:
# 9. Inspect the Pareto table and plot accuracy vs. size vs. latency
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("reports/pareto_results.csv")
display(df)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(df["size_kb"], df["macro_f1"])
for _, r in df.iterrows():
    axes[0].annotate(r["model"], (r["size_kb"], r["macro_f1"]), fontsize=7)
axes[0].set_xlabel("Size (KB)"); axes[0].set_ylabel("Macro F1"); axes[0].set_title("Size vs. Macro F1")

axes[1].scatter(df["latency_ms"], df["macro_f1"])
for _, r in df.iterrows():
    axes[1].annotate(r["model"], (r["latency_ms"], r["macro_f1"]), fontsize=7)
axes[1].set_xlabel("CPU Latency (ms)"); axes[1].set_ylabel("Macro F1"); axes[1].set_title("Latency vs. Macro F1")
plt.tight_layout(); plt.savefig("reports/pareto_plot.png", dpi=150); plt.show()

# Critical-class (mel/akiec) degradation check
print(df[["model", "macro_f1", "f1_mel", "f1_akiec"]])

In [ ]:
# 10. Zip up models + reports so you can download everything back out of Colab
!zip -r results_bundle.zip models reports
from google.colab import files
files.download("results_bundle.zip")